In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

data = [
    (1, "Anitha", "Electronics", 25000, "2026-01-10"),
    (2, "Rahul", "Grocery", 5000, "2026-01-11"),
    (3, "Priya", "Electronics", None, "2026-01-11"),
    (4, "Kiran", "Fashion", 12000, "2026-01-12"),
    (5, "Anitha", "Electronics", 25000, "2026-01-10"),
    (6, None, "Grocery", 7000, "2026-01-13"),
    (7, "Vijay", "Fashion", -2000, "2026-01-14"),
    (8, "Meena", "Electronics", 18000, None)
]

schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("customer_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("amount", IntegerType(), True),
    StructField("purchase_date", StringType(), True)
])

df = spark.createDataFrame(data, schema)

display(df)

customer_id,customer_name,category,amount,purchase_date
1,Anitha,Electronics,25000,2026-01-10
2,Rahul,Grocery,5000,2026-01-11
3,Priya,Electronics,null,2026-01-11
4,Kiran,Fashion,12000,2026-01-12
5,Anitha,Electronics,25000,2026-01-10
6,null,Grocery,7000,2026-01-13
7,Vijay,Fashion,-2000,2026-01-14
8,Meena,Electronics,18000,null


In [0]:
df_clean = (
    df
    .dropDuplicates()
    .filter(col("customer_name").isNotNull())
    .fillna({"amount": 0})
    .filter(col("amount") >= 0)
    .fillna({"purchase_date": "2026-01-15"})
)

display(df_clean)

customer_id,customer_name,category,amount,purchase_date
1,Anitha,Electronics,25000,2026-01-10
2,Rahul,Grocery,5000,2026-01-11
3,Priya,Electronics,0,2026-01-11
4,Kiran,Fashion,12000,2026-01-12
5,Anitha,Electronics,25000,2026-01-10
8,Meena,Electronics,18000,2026-01-15


In [0]:
df_transformed = (
    df_clean
    .withColumn(
        "purchase_date",
        to_date(col("purchase_date"), "yyyy-MM-dd")
    )
    .withColumn(
        "gst_amount",
        round(col("amount") * 0.18, 2)
    )
    .withColumn(
        "total_amount",
        col("amount") + col("gst_amount")
    )
)

display(df_transformed)

customer_id,customer_name,category,amount,purchase_date,gst_amount,total_amount
1,Anitha,Electronics,25000,2026-01-10,4500.0,29500.0
2,Rahul,Grocery,5000,2026-01-11,900.0,5900.0
3,Priya,Electronics,0,2026-01-11,0.0,0.0
4,Kiran,Fashion,12000,2026-01-12,2160.0,14160.0
5,Anitha,Electronics,25000,2026-01-10,4500.0,29500.0
8,Meena,Electronics,18000,2026-01-15,3240.0,21240.0


In [0]:
agg_df = (
    df_transformed
    .groupBy("category")
    .agg(
        sum("amount").alias("total_sales"),
        avg("amount").alias("average_sales"),
        count("*").alias("total_transactions")
    )
)

display(agg_df)

category,total_sales,average_sales,total_transactions
Fashion,12000,12000.0,1
Grocery,5000,5000.0,1
Electronics,68000,17000.0,4


In [0]:
# Save transformed data as CSV
df_transformed.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("dbfs:/FileStore/transformed_data_csv")

# Save aggregated data as CSV
agg_df.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("dbfs:/FileStore/aggregated_data_csv")